In [1]:
import streamlit

print("Streamlit version:", streamlit.__version__)

Streamlit version: 1.51.0


In [2]:
from pathlib import Path

app_folder = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app"
)

app_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Application folder created:")
print(app_folder)

Application folder created:
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app


In [3]:
from pathlib import Path
import shutil

source_folder = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
)

app_folder = source_folder / "streamlit_app"

required_files = [
    "best_emi_regression_model.pkl",
    "classification_preprocessor.pkl",
    "random_forest_classifier.pkl",
    "xgboost_classifier.pkl",
    "xgb_label_encoder.pkl",
    "final_feature_importance.csv"
]

for filename in required_files:

    source_file = source_folder / filename
    destination_file = app_folder / filename

    if source_file.exists():

        shutil.copy2(
            source_file,
            destination_file
        )

        print("Copied:", filename)

    else:

        print("NOT FOUND:", filename)

Copied: best_emi_regression_model.pkl
Copied: classification_preprocessor.pkl
Copied: random_forest_classifier.pkl
Copied: xgboost_classifier.pkl
Copied: xgb_label_encoder.pkl
Copied: final_feature_importance.csv


In [4]:
print("=" * 60)
print("STREAMLIT APPLICATION FILES")
print("=" * 60)

for file in sorted(app_folder.iterdir()):
    print(file.name)

STREAMLIT APPLICATION FILES
__pycache__
app.py
best_emi_regression_model.pkl
classification_preprocessor.pkl
final_feature_importance.csv
random_forest_classifier.pkl
xgb_label_encoder.pkl
xgboost_classifier.pkl


In [5]:
app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="EMI Prediction System",
    page_icon="💰",
    layout="wide"
)


# ============================================================
# APPLICATION DIRECTORY
# ============================================================

APP_DIR = Path(__file__).resolve().parent


# ============================================================
# FILE PATHS
# ============================================================

REGRESSION_MODEL_PATH = (
    APP_DIR / "best_emi_regression_model.pkl"
)

CLASSIFICATION_PREPROCESSOR_PATH = (
    APP_DIR / "classification_preprocessor.pkl"
)

RF_CLASSIFIER_PATH = (
    APP_DIR / "random_forest_classifier.pkl"
)

XGB_CLASSIFIER_PATH = (
    APP_DIR / "xgboost_classifier.pkl"
)

LABEL_ENCODER_PATH = (
    APP_DIR / "xgb_label_encoder.pkl"
)

FEATURE_IMPORTANCE_PATH = (
    APP_DIR / "final_feature_importance.csv"
)


# ============================================================
# LOAD MODELS
# ============================================================

@st.cache_resource
def load_models():

    regression_model = joblib.load(
        REGRESSION_MODEL_PATH
    )

    classification_preprocessor = joblib.load(
        CLASSIFICATION_PREPROCESSOR_PATH
    )

    rf_classifier = joblib.load(
        RF_CLASSIFIER_PATH
    )

    xgb_classifier = joblib.load(
        XGB_CLASSIFIER_PATH
    )

    label_encoder = joblib.load(
        LABEL_ENCODER_PATH
    )

    return (
        regression_model,
        classification_preprocessor,
        rf_classifier,
        xgb_classifier,
        label_encoder
    )


# ============================================================
# LOAD FEATURE IMPORTANCE
# ============================================================

@st.cache_data
def load_feature_importance():

    if FEATURE_IMPORTANCE_PATH.exists():

        return pd.read_csv(
            FEATURE_IMPORTANCE_PATH
        )

    return pd.DataFrame()


# ============================================================
# SAFE MODEL LOADING
# ============================================================

try:

    (
        regression_model,
        classification_preprocessor,
        rf_classifier,
        xgb_classifier,
        label_encoder
    ) = load_models()

    models_loaded = True

except Exception as e:

    models_loaded = False
    model_error = str(e)


feature_importance_df = load_feature_importance()


# ============================================================
# TITLE
# ============================================================

st.title(
    "💰 EMI Prediction & Loan Risk Analysis System"
)

st.write(
    "Machine Learning based EMI prediction and "
    "loan risk classification system."
)


# ============================================================
# MODEL STATUS
# ============================================================

if models_loaded:

    st.success(
        "✅ All machine learning models loaded successfully."
    )

else:

    st.error(
        "❌ Model loading failed."
    )

    st.code(
        model_error
    )

    st.stop()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "Application Menu"
)

page = st.sidebar.radio(
    "Select Module",
    [
        "EMI Prediction",
        "Loan Risk Classification",
        "Feature Importance"
    ]
)


# ============================================================
# EMI PREDICTION
# ============================================================

if page == "EMI Prediction":

    st.header(
        "💳 Maximum Monthly EMI Prediction"
    )

    st.write(
        "Enter the applicant financial information."
    )

    col1, col2 = st.columns(2)

    with col1:

        monthly_income = st.number_input(
            "Monthly Income",
            min_value=0.0,
            value=50000.0,
            step=1000.0
        )

        available_income = st.number_input(
            "Available Monthly Income",
            min_value=0.0,
            value=25000.0,
            step=1000.0
        )

        current_emi = st.number_input(
            "Current EMI Amount",
            min_value=0.0,
            value=5000.0,
            step=500.0
        )

        expense_ratio = st.number_input(
            "Expense to Income Ratio",
            min_value=0.0,
            max_value=1.0,
            value=0.30,
            step=0.01
        )

    with col2:

        bank_balance = st.number_input(
            "Bank Balance",
            min_value=0.0,
            value=100000.0,
            step=5000.0
        )

        credit_score = st.number_input(
            "Credit Score",
            min_value=300.0,
            max_value=900.0,
            value=700.0,
            step=1.0
        )

        years_employment = st.number_input(
            "Years of Employment",
            min_value=0.0,
            value=5.0,
            step=1.0
        )

        debt_income_ratio = st.number_input(
            "Debt to Income Ratio",
            min_value=0.0,
            max_value=2.0,
            value=0.30,
            step=0.01
        )

    if st.button(
        "🔮 Predict Maximum EMI",
        type="primary"
    ):

        input_data = pd.DataFrame({

            "monthly_income": [
                monthly_income
            ],

            "available_monthly_income": [
                available_income
            ],

            "current_emi_amount": [
                current_emi
            ],

            "expense_to_income_ratio": [
                expense_ratio
            ],

            "bank_balance": [
                bank_balance
            ],

            "credit_score": [
                credit_score
            ],

            "years_of_employment": [
                years_employment
            ],

            "debt_to_income_ratio": [
                debt_income_ratio
            ]

        })

        try:

            prediction = regression_model.predict(
                input_data
            )

            predicted_emi = float(
                np.asarray(prediction).ravel()[0]
            )

            st.success(
                f"Predicted Maximum Monthly EMI: ₹{predicted_emi:,.2f}"
            )

        except Exception as e:

            st.error(
                "Prediction could not be generated."
            )

            st.write(
                "The saved regression model expects a "
                "different input structure."
            )

            st.code(
                str(e)
            )


# ============================================================
# LOAN RISK CLASSIFICATION
# ============================================================

elif page == "Loan Risk Classification":

    st.header(
        "📊 Loan Risk Classification"
    )

    st.write(
        "Estimate the applicant's loan risk category."
    )

    col1, col2 = st.columns(2)

    with col1:

        income = st.number_input(
            "Applicant Monthly Income",
            min_value=0.0,
            value=50000.0,
            step=1000.0,
            key="classification_income"
        )

        credit = st.number_input(
            "Credit Score",
            min_value=300.0,
            max_value=900.0,
            value=700.0,
            step=1.0,
            key="classification_credit"
        )

        balance = st.number_input(
            "Bank Balance",
            min_value=0.0,
            value=100000.0,
            step=5000.0
        )

    with col2:

        emi = st.number_input(
            "Current EMI",
            min_value=0.0,
            value=5000.0,
            step=500.0
        )

        loans = st.selectbox(
            "Existing Loans",
            [
                "No",
                "Yes"
            ]
        )

        employment = st.number_input(
            "Years of Employment",
            min_value=0.0,
            value=5.0,
            step=1.0
        )

    if st.button(
        "🔍 Classify Loan Risk",
        type="primary"
    ):

        classification_input = pd.DataFrame({

            "monthly_income": [
                income
            ],

            "credit_score": [
                credit
            ],

            "bank_balance": [
                balance
            ],

            "current_emi_amount": [
                emi
            ],

            "existing_loans": [
                loans
            ],

            "years_of_employment": [
                employment
            ]

        })

        try:

            transformed_data = (
                classification_preprocessor.transform(
                    classification_input
                )
            )

            prediction = rf_classifier.predict(
                transformed_data
            )

            if hasattr(
                label_encoder,
                "inverse_transform"
            ):

                prediction_label = (
                    label_encoder.inverse_transform(
                        prediction
                    )[0]
                )

            else:

                prediction_label = prediction[0]

            st.success(
                f"Predicted Loan Risk: {prediction_label}"
            )

        except Exception as e:

            st.error(
                "Classification could not be generated."
            )

            st.code(
                str(e)
            )


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

elif page == "Feature Importance":

    st.header(
        "📈 Feature Importance Analysis"
    )

    if feature_importance_df.empty:

        st.warning(
            "Feature importance file was not found."
        )

    else:

        st.dataframe(
            feature_importance_df,
            use_container_width=True
        )

        if (
            "Feature" in feature_importance_df.columns
            and
            "Importance" in feature_importance_df.columns
        ):

            chart_data = (
                feature_importance_df
                .sort_values(
                    "Importance",
                    ascending=False
                )
                .head(15)
                .set_index("Feature")
            )

            st.bar_chart(
                chart_data["Importance"]
            )


# ============================================================
# FOOTER
# ============================================================

st.sidebar.markdown("---")

st.sidebar.info(
    "EMI Prediction System\n\n"
    "Developed using Python, "
    "Scikit-learn and Streamlit."
)
'''


app_file = app_folder / "app.py"

app_file.write_text(
    app_code,
    encoding="utf-8"
)

print("app.py created successfully:")
print(app_file)

app.py created successfully:
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\app.py


In [6]:
print(app_file.exists())
print(app_file)

True
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\app.py


In [7]:
import joblib
from pathlib import Path

app_folder = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app"
)

regression_model = joblib.load(
    app_folder / "best_emi_regression_model.pkl"
)

classification_preprocessor = joblib.load(
    app_folder / "classification_preprocessor.pkl"
)

rf_classifier = joblib.load(
    app_folder / "random_forest_classifier.pkl"
)

print("=" * 70)
print("REGRESSION MODEL")
print("=" * 70)

print(
    "Model type:",
    type(regression_model).__name__
)

if hasattr(
    regression_model,
    "feature_names_in_"
):

    print(
        "Regression input columns:"
    )

    print(
        list(
            regression_model.feature_names_in_
        )
    )

else:

    print(
        "Regression model does not expose feature_names_in_."
    )


print()
print("=" * 70)
print("CLASSIFICATION PREPROCESSOR")
print("=" * 70)

print(
    "Preprocessor type:",
    type(classification_preprocessor).__name__
)

if hasattr(
    classification_preprocessor,
    "feature_names_in_"
):

    print(
        "Classification input columns:"
    )

    print(
        list(
            classification_preprocessor.feature_names_in_
        )
    )

else:

    print(
        "Preprocessor does not expose feature_names_in_."
    )


print()
print("=" * 70)
print("CLASSIFIER")
print("=" * 70)

print(
    "Random Forest:",
    type(rf_classifier).__name__
)

if hasattr(
    rf_classifier,
    "n_features_in_"
):

    print(
        "Expected transformed features:",
        rf_classifier.n_features_in_
    )

REGRESSION MODEL
Model type: XGBRegressor
Regression model does not expose feature_names_in_.

CLASSIFICATION PREPROCESSOR
Preprocessor type: ColumnTransformer
Classification input columns:
['age', 'gender', 'marital_status', 'education', 'monthly_salary', 'employment_type', 'years_of_employment', 'company_type', 'house_type', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'emi_scenario', 'requested_amount', 'requested_tenure', 'total_monthly_expenses', 'total_monthly_obligations', 'debt_to_income_ratio', 'expense_to_income_ratio', 'obligation_to_income_ratio', 'available_monthly_income', 'affordability_ratio', 'estimated_requested_emi', 'loan_to_income_ratio', 'projected_emi_burden', 'projected_emi_to_income_ratio', 'emergency_fund_coverage_months', 'bank_balance_to_salary_ratio', 'employment_stabilit

In [8]:
# ============================================================
# STEP 8A
# CHECK ACTUAL EMI DATASET COLUMNS
# ============================================================

import pandas as pd
from pathlib import Path

dataset_paths = [
    Path(r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\EMI_dataset_feature_engineered.csv"),
    Path(r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\EMI_dataset_cleaned.csv"),
    Path(r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\emi_prediction_dataset.csv"),
]

dataset_df = None
dataset_path_used = None

for path in dataset_paths:

    if path.exists():

        try:

            temp_df = pd.read_csv(path)

            print("FOUND:")
            print(path)
            print("Shape:", temp_df.shape)

            dataset_df = temp_df
            dataset_path_used = path

            break

        except Exception as e:

            print("Could not read:", path)
            print(e)


print()
print("=" * 70)
print("DATASET COLUMN CHECK")
print("=" * 70)

if dataset_df is None:

    print("❌ No EMI dataset could be loaded.")

else:

    print(
        "Dataset used:",
        dataset_path_used
    )

    print(
        "Number of columns:",
        len(dataset_df.columns)
    )

    print()

    for i, column in enumerate(
        dataset_df.columns,
        start=1
    ):

        print(
            f"{i:02d}. {column}"
        )

FOUND:
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\EMI_dataset_feature_engineered.csv
Shape: (404342, 45)

DATASET COLUMN CHECK
Dataset used: C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\EMI_dataset_feature_engineered.csv
Number of columns: 45

01. age
02. gender
03. marital_status
04. education
05. monthly_salary
06. employment_type
07. years_of_employment
08. company_type
09. house_type
10. monthly_rent
11. family_size
12. dependents
13. school_fees
14. college_fees
15. travel_expenses
16. groceries_utilities
17. other_monthly_expenses
18. existing_loans
19. current_emi_amount
20. credit_score
21. bank_balance
22. emergency_fund
23. emi_scenario
24. requested_amount
25. requested_tenure
26. emi_eligibility
27. max_monthly_emi
28. total_monthly_expenses
29. total_monthly_obligations
30. debt_to_income_ratio
31. expense_to_income_ratio
32. obligation_to_income_ratio
33. available_monthly_income
34. affordability_ratio
35. estimated_requested_emi
36. loan_to_income_ratio
37. projected_emi_

In [9]:
# ============================================================
# STEP 8B
# INSPECT SAVED REGRESSION MODEL
# ============================================================

import joblib
from pathlib import Path

regression_model_path = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\best_emi_regression_model.pkl"
)

regression_model = joblib.load(
    regression_model_path
)

print("=" * 70)
print("REGRESSION MODEL INSPECTION")
print("=" * 70)

print(
    "Model type:",
    type(regression_model)
)

print()

print(
    "Number of expected input features:",
    getattr(
        regression_model,
        "n_features_in_",
        "Not available"
    )
)

print()

print(
    "Model parameters:"
)

print(
    regression_model.get_params()
)

REGRESSION MODEL INSPECTION
Model type: <class 'xgboost.sklearn.XGBRegressor'>

Number of expected input features: 74

Model parameters:
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'rmse', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': N

In [10]:
# ============================================================
# STEP 8C
# INSPECT XGBOOST REGRESSION FEATURE INFORMATION
# ============================================================

print("=" * 70)
print("XGBOOST REGRESSION FEATURE INFORMATION")
print("=" * 70)

print(
    "Model type:",
    type(regression_model).__name__
)

print(
    "Expected features:",
    regression_model.n_features_in_
)

# ------------------------------------------------------------
# Check booster feature names
# ------------------------------------------------------------

booster = regression_model.get_booster()

print()
print("Booster feature names available:")

booster_features = booster.feature_names

if booster_features is None:

    print("❌ Booster feature names are not available.")

else:

    print(
        "Number of booster features:",
        len(booster_features)
    )

    print()

    for i, feature in enumerate(
        booster_features,
        start=1
    ):

        print(
            f"{i:02d}. {feature}"
        )

XGBOOST REGRESSION FEATURE INFORMATION
Model type: XGBRegressor
Expected features: 74

Booster feature names available:
❌ Booster feature names are not available.


In [11]:
# ============================================================
# STEP 8D
# SEARCH ALL SAVED PKL FILES FOR PREPROCESSORS
# ============================================================

from pathlib import Path
import joblib

emi_folder = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
)

pkl_files = list(
    emi_folder.glob("*.pkl")
)

print("=" * 70)
print("AVAILABLE PKL FILES")
print("=" * 70)

for file in pkl_files:

    print()
    print("FILE:", file.name)

    try:

        obj = joblib.load(file)

        print(
            "OBJECT TYPE:",
            type(obj).__name__
        )

        if hasattr(
            obj,
            "feature_names_in_"
        ):

            print(
                "feature_names_in_:",
                len(obj.feature_names_in_)
            )

        if hasattr(
            obj,
            "transformers_"
        ):

            print(
                "Contains transformers: YES"
            )

        if hasattr(
            obj,
            "n_features_in_"
        ):

            print(
                "n_features_in_:",
                obj.n_features_in_
            )

    except Exception as e:

        print(
            "Could not inspect:",
            str(e)
        )

AVAILABLE PKL FILES

FILE: best_emi_regression_model.pkl
OBJECT TYPE: XGBRegressor
n_features_in_: 74

FILE: classification_preprocessor.pkl
OBJECT TYPE: ColumnTransformer
feature_names_in_: 43
Contains transformers: YES
n_features_in_: 43

FILE: random_forest_classifier.pkl
OBJECT TYPE: RandomForestClassifier
n_features_in_: 74

FILE: xgboost_classifier.pkl
OBJECT TYPE: XGBClassifier
n_features_in_: 74

FILE: xgb_label_encoder.pkl
OBJECT TYPE: LabelEncoder


In [12]:
# ============================================================
# STEP 8E
# SEARCH PROJECT FILES FOR PREPROCESSOR / PIPELINE FILES
# ============================================================

all_files = list(
    emi_folder.rglob("*")
)

print("=" * 70)
print("POSSIBLE PREPROCESSING / PIPELINE FILES")
print("=" * 70)

keywords = [
    "preprocess",
    "processor",
    "pipeline",
    "transform",
    "encoder",
    "scaler",
    "regression"
]

found = []

for file in all_files:

    if not file.is_file():
        continue

    name = file.name.lower()

    if any(
        keyword in name
        for keyword in keywords
    ):

        found.append(file)


if len(found) == 0:

    print(
        "No additional preprocessing files found."
    )

else:

    for file in found:

        print(file)

POSSIBLE PREPROCESSING / PIPELINE FILES
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\best_emi_regression_model.pkl
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\classification_preprocessor.pkl
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\emi_regression_results.csv
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\final_regression_model_summary.csv
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\final_regression_predictions.csv
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\xgb_label_encoder.pkl
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\best_emi_regression_model.pkl
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\classification_preprocessor.pkl
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\xgb_label_encoder.pkl


In [13]:
# ============================================================
# STEP 8F
# IDENTIFY CLASSIFICATION INPUT AND TARGET COLUMNS
# ============================================================

import pandas as pd
import joblib

dataset_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\EMI_dataset_feature_engineered.csv"
)

preprocessor_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\classification_preprocessor.pkl"
)

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

dataset = pd.read_csv(dataset_path)

# ------------------------------------------------------------
# Load classification preprocessor
# ------------------------------------------------------------

classification_preprocessor = joblib.load(
    preprocessor_path
)

dataset_columns = list(
    dataset.columns
)

preprocessor_columns = list(
    classification_preprocessor.feature_names_in_
)

print("=" * 70)
print("CLASSIFICATION COLUMN ANALYSIS")
print("=" * 70)

print()
print("Dataset columns:", len(dataset_columns))

print(
    "Preprocessor input columns:",
    len(preprocessor_columns)
)

# ------------------------------------------------------------
# Columns used by classification preprocessor
# ------------------------------------------------------------

print()
print("=" * 70)
print("COLUMNS USED BY CLASSIFICATION PREPROCESSOR")
print("=" * 70)

for i, column in enumerate(
    preprocessor_columns,
    start=1
):

    print(
        f"{i:02d}. {column}"
    )

# ------------------------------------------------------------
# Dataset columns NOT used by preprocessor
# ------------------------------------------------------------

unused_columns = [
    column
    for column in dataset_columns
    if column not in preprocessor_columns
]

print()
print("=" * 70)
print("DATASET COLUMNS NOT USED BY PREPROCESSOR")
print("=" * 70)

for i, column in enumerate(
    unused_columns,
    start=1
):

    print(
        f"{i:02d}. {column}"
    )

# ------------------------------------------------------------
# Verify preprocessing output
# ------------------------------------------------------------

X_classification = dataset[
    preprocessor_columns
].copy()

try:

    transformed = classification_preprocessor.transform(
        X_classification.head(5)
    )

    print()
    print("=" * 70)
    print("PREPROCESSOR TRANSFORMATION CHECK")
    print("=" * 70)

    print(
        "Input shape:",
        X_classification.head(5).shape
    )

    print(
        "Transformed shape:",
        transformed.shape
    )

    print(
        "Expected classifier features:",
        74
    )

    if transformed.shape[1] == 74:

        print(
            "✅ Classification preprocessing produces exactly 74 features."
        )

    else:

        print(
            "⚠️ Unexpected transformed feature count."
        )

except Exception as e:

    print()
    print(
        "❌ Transformation failed:"
    )

    print(e)

CLASSIFICATION COLUMN ANALYSIS

Dataset columns: 45
Preprocessor input columns: 43

COLUMNS USED BY CLASSIFICATION PREPROCESSOR
01. age
02. gender
03. marital_status
04. education
05. monthly_salary
06. employment_type
07. years_of_employment
08. company_type
09. house_type
10. monthly_rent
11. family_size
12. dependents
13. school_fees
14. college_fees
15. travel_expenses
16. groceries_utilities
17. other_monthly_expenses
18. existing_loans
19. current_emi_amount
20. credit_score
21. bank_balance
22. emergency_fund
23. emi_scenario
24. requested_amount
25. requested_tenure
26. total_monthly_expenses
27. total_monthly_obligations
28. debt_to_income_ratio
29. expense_to_income_ratio
30. obligation_to_income_ratio
31. available_monthly_income
32. affordability_ratio
33. estimated_requested_emi
34. loan_to_income_ratio
35. projected_emi_burden
36. projected_emi_to_income_ratio
37. emergency_fund_coverage_months
38. bank_balance_to_salary_ratio
39. employment_stability_score
40. credit_sco

In [14]:
# ============================================================
# STEP 8G
# VERIFY REGRESSION TARGET AND FEATURE RELATIONSHIPS
# ============================================================

import pandas as pd
import numpy as np

dataset_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\EMI_dataset_feature_engineered.csv"
)

regression_df = pd.read_csv(dataset_path)

print("=" * 70)
print("REGRESSION DATASET VERIFICATION")
print("=" * 70)

print()
print("Dataset shape:")
print(regression_df.shape)

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

target_column = "max_monthly_emi"

print()
print("=" * 70)
print("REGRESSION TARGET")
print("=" * 70)

print("Target column:", target_column)

if target_column in regression_df.columns:

    print("✅ Target column exists.")

    print(
        "Target dtype:",
        regression_df[target_column].dtype
    )

    print(
        "Target missing values:",
        regression_df[target_column].isna().sum()
    )

    print(
        "Target minimum:",
        regression_df[target_column].min()
    )

    print(
        "Target maximum:",
        regression_df[target_column].max()
    )

else:

    print("❌ Target column not found.")

# ------------------------------------------------------------
# Check important engineered features
# ------------------------------------------------------------

required_features = [
    "total_monthly_expenses",
    "total_monthly_obligations",
    "debt_to_income_ratio",
    "expense_to_income_ratio",
    "obligation_to_income_ratio",
    "available_monthly_income",
    "affordability_ratio",
    "estimated_requested_emi",
    "loan_to_income_ratio",
    "projected_emi_burden",
    "projected_emi_to_income_ratio",
    "emergency_fund_coverage_months",
    "bank_balance_to_salary_ratio",
    "employment_stability_score",
    "credit_score_normalized",
    "credit_risk_level",
    "dependent_ratio",
    "financial_stress_indicator"
]

print()
print("=" * 70)
print("ENGINEERED FEATURE CHECK")
print("=" * 70)

missing_features = []

for feature in required_features:

    if feature in regression_df.columns:

        print(
            f"✅ {feature}"
        )

    else:

        print(
            f"❌ {feature}"
        )

        missing_features.append(feature)

# ------------------------------------------------------------
# Check data types
# ------------------------------------------------------------

print()
print("=" * 70)
print("DATA TYPES")
print("=" * 70)

display(
    regression_df[
        required_features
    ].dtypes.to_frame(
        name="dtype"
    )
)

# ------------------------------------------------------------
# Regression candidate input columns
# ------------------------------------------------------------

print()
print("=" * 70)
print("REGRESSION INPUT CANDIDATES")
print("=" * 70)

regression_input_columns = [
    column
    for column in regression_df.columns
    if column != target_column
]

print(
    "Total candidate input columns:",
    len(regression_input_columns)
)

for i, column in enumerate(
    regression_input_columns,
    start=1
):

    print(
        f"{i:02d}. {column}"
    )

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print()
print("=" * 70)
print("STEP 8G STATUS")
print("=" * 70)

if (
    target_column in regression_df.columns
    and len(missing_features) == 0
):

    print(
        "✅ Regression target and engineered features verified."
    )

else:

    print(
        "⚠️ Some regression information is missing."
    )

REGRESSION DATASET VERIFICATION

Dataset shape:
(404342, 45)

REGRESSION TARGET
Target column: max_monthly_emi
✅ Target column exists.
Target dtype: float64
Target missing values: 0
Target minimum: 500.0
Target maximum: 50000.0

ENGINEERED FEATURE CHECK
✅ total_monthly_expenses
✅ total_monthly_obligations
✅ debt_to_income_ratio
✅ expense_to_income_ratio
✅ obligation_to_income_ratio
✅ available_monthly_income
✅ affordability_ratio
✅ estimated_requested_emi
✅ loan_to_income_ratio
✅ projected_emi_burden
✅ projected_emi_to_income_ratio
✅ emergency_fund_coverage_months
✅ bank_balance_to_salary_ratio
✅ employment_stability_score
✅ credit_score_normalized
✅ credit_risk_level
✅ dependent_ratio
✅ financial_stress_indicator

DATA TYPES


,dtype
total_monthly_expenses,float64
total_monthly_obligations,float64
debt_to_income_ratio,float64
expense_to_income_ratio,float64
obligation_to_income_ratio,float64
available_monthly_income,float64
affordability_ratio,float64
estimated_requested_emi,float64
loan_to_income_ratio,float64
projected_emi_burden,float64



REGRESSION INPUT CANDIDATES
Total candidate input columns: 44
01. age
02. gender
03. marital_status
04. education
05. monthly_salary
06. employment_type
07. years_of_employment
08. company_type
09. house_type
10. monthly_rent
11. family_size
12. dependents
13. school_fees
14. college_fees
15. travel_expenses
16. groceries_utilities
17. other_monthly_expenses
18. existing_loans
19. current_emi_amount
20. credit_score
21. bank_balance
22. emergency_fund
23. emi_scenario
24. requested_amount
25. requested_tenure
26. emi_eligibility
27. total_monthly_expenses
28. total_monthly_obligations
29. debt_to_income_ratio
30. expense_to_income_ratio
31. obligation_to_income_ratio
32. available_monthly_income
33. affordability_ratio
34. estimated_requested_emi
35. loan_to_income_ratio
36. projected_emi_burden
37. projected_emi_to_income_ratio
38. emergency_fund_coverage_months
39. bank_balance_to_salary_ratio
40. employment_stability_score
41. credit_score_normalized
42. credit_risk_level
43. depen

In [15]:
# ============================================================
# STEP 8H
# CHECK CLASSIFICATION TRANSFORMER STRUCTURE
# ============================================================

import joblib

preprocessor_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\classification_preprocessor.pkl"
)

cp = joblib.load(
    preprocessor_path
)

print("=" * 70)
print("CLASSIFICATION PREPROCESSOR STRUCTURE")
print("=" * 70)

print()
print(
    "Preprocessor:",
    type(cp).__name__
)

print()
print("Transformers:")

for name, transformer, columns in cp.transformers_:

    print()
    print("Transformer name:", name)
    print(
        "Transformer type:",
        type(transformer).__name__
    )

    if isinstance(columns, list):

        print(
            "Number of columns:",
            len(columns)
        )

        print(
            "Columns:",
            columns
        )

    else:

        print(
            "Columns:",
            columns
        )

# ------------------------------------------------------------
# Output feature names
# ------------------------------------------------------------

print()
print("=" * 70)
print("TRANSFORMED FEATURE NAMES")
print("=" * 70)

try:

    transformed_names = (
        cp.get_feature_names_out()
    )

    print(
        "Number of transformed features:",
        len(transformed_names)
    )

    for i, feature in enumerate(
        transformed_names,
        start=1
    ):

        print(
            f"{i:02d}. {feature}"
        )

except Exception as e:

    print(
        "Could not obtain transformed feature names."
    )

    print(e)

CLASSIFICATION PREPROCESSOR STRUCTURE

Preprocessor: ColumnTransformer

Transformers:

Transformer name: numerical
Transformer type: Pipeline
Number of columns: 33
Columns: ['age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure', 'total_monthly_expenses', 'total_monthly_obligations', 'debt_to_income_ratio', 'expense_to_income_ratio', 'obligation_to_income_ratio', 'available_monthly_income', 'affordability_ratio', 'estimated_requested_emi', 'loan_to_income_ratio', 'projected_emi_burden', 'projected_emi_to_income_ratio', 'emergency_fund_coverage_months', 'bank_balance_to_salary_ratio', 'employment_stability_score', 'credit_score_normalized', 'dependent_ratio']

Transformer name: categorical
Transformer type: Pipeline
Number of columns: 10
Colum

In [16]:
# ============================================================
# STEP 8I
# VERIFY REGRESSION MODEL + CLASSIFICATION PREPROCESSOR
# ============================================================

import pandas as pd
import joblib
import numpy as np

print("=" * 70)
print("REGRESSION + PREPROCESSOR COMPATIBILITY TEST")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

dataset_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\EMI_dataset_feature_engineered.csv"
)

preprocessor_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\classification_preprocessor.pkl"
)

regression_model_path = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\best_emi_regression_model.pkl"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

df_test = pd.read_csv(
    dataset_path
)

preprocessor = joblib.load(
    preprocessor_path
)

regression_model = joblib.load(
    regression_model_path
)

print()
print("Dataset loaded:", df_test.shape)

print(
    "Preprocessor:",
    type(preprocessor).__name__
)

print(
    "Regression model:",
    type(regression_model).__name__
)

# ------------------------------------------------------------
# Get classification input columns
# ------------------------------------------------------------

input_columns = list(
    preprocessor.feature_names_in_
)

print()
print(
    "Preprocessor input columns:",
    len(input_columns)
)

# ------------------------------------------------------------
# Take small sample
# ------------------------------------------------------------

X_sample = df_test[
    input_columns
].head(5).copy()

y_sample = df_test[
    "max_monthly_emi"
].head(5).copy()

print()
print(
    "Sample input shape:",
    X_sample.shape
)

# ------------------------------------------------------------
# Transform
# ------------------------------------------------------------

try:

    X_transformed = preprocessor.transform(
        X_sample
    )

    print()
    print(
        "Transformed shape:",
        X_transformed.shape
    )

except Exception as e:

    print()
    print(
        "❌ PREPROCESSING FAILED"
    )

    print(
        str(e)
    )

    X_transformed = None

# ------------------------------------------------------------
# Test regression model
# ------------------------------------------------------------

if X_transformed is not None:

    try:

        predictions = regression_model.predict(
            X_transformed
        )

        print()
        print(
            "=" * 70
        )

        print(
            "REGRESSION PREDICTION TEST"
        )

        print(
            "=" * 70
        )

        print(
            "Predictions generated successfully."
        )

        print(
            "Prediction shape:",
            predictions.shape
        )

        print()

        print(
            "Actual values:"
        )

        print(
            y_sample.to_numpy()
        )

        print()

        print(
            "Predicted values:"
        )

        print(
            predictions
        )

        print()

        print(
            "✅ REGRESSION MODEL ACCEPTED THE 74 TRANSFORMED FEATURES."
        )

    except Exception as e:

        print()
        print(
            "=" * 70
        )

        print(
            "❌ REGRESSION MODEL REJECTED THE TRANSFORMED FEATURES"
        )

        print(
            "=" * 70
        )

        print(
            str(e)
        )

REGRESSION + PREPROCESSOR COMPATIBILITY TEST

Dataset loaded: (404342, 45)
Preprocessor: ColumnTransformer
Regression model: XGBRegressor

Preprocessor input columns: 43

Sample input shape: (5, 43)

Transformed shape: (5, 74)

REGRESSION PREDICTION TEST
Predictions generated successfully.
Prediction shape: (5,)

Actual values:
[  500.   700. 27775. 16170.   500.]

Predicted values:
[  408.16025   705.0086  28376.043   16465.088     733.0131 ]

✅ REGRESSION MODEL ACCEPTED THE 74 TRANSFORMED FEATURES.


In [17]:
# ============================================================
# STEP 10
# CREATE STREAMLIT APPLICATION
# ============================================================

from pathlib import Path

app_folder = Path(
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app"
)

app_folder.mkdir(
    parents=True,
    exist_ok=True
)

app_path = app_folder / "app.py"

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="EMI Prediction System",
    page_icon="💰",
    layout="wide"
)


# ============================================================
# PATH CONFIGURATION
# ============================================================

BASE_DIR = Path(__file__).resolve().parent


REGRESSION_MODEL_PATH = (
    BASE_DIR / "best_emi_regression_model.pkl"
)

PREPROCESSOR_PATH = (
    BASE_DIR / "classification_preprocessor.pkl"
)

RF_MODEL_PATH = (
    BASE_DIR / "random_forest_classifier.pkl"
)

XGB_MODEL_PATH = (
    BASE_DIR / "xgboost_classifier.pkl"
)

LABEL_ENCODER_PATH = (
    BASE_DIR / "xgb_label_encoder.pkl"
)


# ============================================================
# LOAD MODELS
# ============================================================

@st.cache_resource
def load_models():

    regression_model = joblib.load(
        REGRESSION_MODEL_PATH
    )

    preprocessor = joblib.load(
        PREPROCESSOR_PATH
    )

    rf_model = joblib.load(
        RF_MODEL_PATH
    )

    xgb_model = joblib.load(
        XGB_MODEL_PATH
    )

    label_encoder = joblib.load(
        LABEL_ENCODER_PATH
    )

    return (
        regression_model,
        preprocessor,
        rf_model,
        xgb_model,
        label_encoder
    )


try:

    (
        regression_model,
        preprocessor,
        rf_model,
        xgb_model,
        label_encoder
    ) = load_models()

    models_loaded = True

except Exception as e:

    models_loaded = False

    st.error(
        "Model loading failed."
    )

    st.exception(e)


# ============================================================
# TITLE
# ============================================================

st.title(
    "💰 EMI Prediction & Financial Risk Analysis"
)

st.markdown(
    """
    ### AI-Powered EMI Decision Support System

    Enter the applicant's financial information below to estimate
    the maximum affordable monthly EMI and analyse loan eligibility.
    """
)


if not models_loaded:

    st.stop()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "Application Information"
)

st.sidebar.info(
    """
    This application uses trained Machine Learning models
    developed for the EMI Prediction Project.
    """
)

st.sidebar.markdown(
    """
    **Models**

    • XGBoost Regression  
    • Random Forest Classification  
    • XGBoost Classification
    """
)


# ============================================================
# PERSONAL INFORMATION
# ============================================================

st.header(
    "👤 Personal Information"
)

col1, col2, col3 = st.columns(3)


with col1:

    age = st.number_input(
        "Age",
        min_value=18,
        max_value=80,
        value=30,
        step=1
    )


with col2:

    gender = st.selectbox(
        "Gender",
        [
            "Male",
            "Female"
        ]
    )


with col3:

    marital_status = st.selectbox(
        "Marital Status",
        [
            "Single",
            "Married"
        ]
    )


col1, col2, col3 = st.columns(3)


with col1:

    education = st.selectbox(
        "Education",
        [
            "High School",
            "Graduate",
            "Post Graduate",
            "Professional"
        ]
    )


with col2:

    family_size = st.number_input(
        "Family Size",
        min_value=1,
        max_value=20,
        value=4,
        step=1
    )


with col3:

    dependents = st.number_input(
        "Dependents",
        min_value=0,
        max_value=15,
        value=2,
        step=1
    )


# ============================================================
# EMPLOYMENT
# ============================================================

st.header(
    "💼 Employment Information"
)

col1, col2, col3 = st.columns(3)


with col1:

    monthly_salary = st.number_input(
        "Monthly Salary (₹)",
        min_value=5000.0,
        max_value=10000000.0,
        value=50000.0,
        step=1000.0
    )


with col2:

    employment_type = st.selectbox(
        "Employment Type",
        [
            "Private",
            "Government",
            "Self-employed"
        ]
    )


with col3:

    years_of_employment = st.number_input(
        "Years of Employment",
        min_value=0.0,
        max_value=50.0,
        value=5.0,
        step=0.5
    )


col1, col2 = st.columns(2)


with col1:

    company_type = st.selectbox(
        "Company Type",
        [
            "MNC",
            "Large Indian",
            "Mid-size",
            "Small",
            "Startup"
        ]
    )


with col2:

    house_type = st.selectbox(
        "House Type",
        [
            "Own",
            "Rented",
            "Family"
        ]
    )


# ============================================================
# MONTHLY EXPENSES
# ============================================================

st.header(
    "🏠 Monthly Expenses"
)

col1, col2, col3 = st.columns(3)


with col1:

    monthly_rent = st.number_input(
        "Monthly Rent (₹)",
        min_value=0.0,
        max_value=1000000.0,
        value=10000.0,
        step=500.0
    )


with col2:

    school_fees = st.number_input(
        "School Fees (₹)",
        min_value=0.0,
        max_value=500000.0,
        value=3000.0,
        step=500.0
    )


with col3:

    college_fees = st.number_input(
        "College Fees (₹)",
        min_value=0.0,
        max_value=500000.0,
        value=0.0,
        step=500.0
    )


col1, col2, col3 = st.columns(3)


with col1:

    travel_expenses = st.number_input(
        "Travel Expenses (₹)",
        min_value=0.0,
        max_value=500000.0,
        value=3000.0,
        step=500.0
    )


with col2:

    groceries_utilities = st.number_input(
        "Groceries & Utilities (₹)",
        min_value=0.0,
        max_value=500000.0,
        value=8000.0,
        step=500.0
    )


with col3:

    other_monthly_expenses = st.number_input(
        "Other Monthly Expenses (₹)",
        min_value=0.0,
        max_value=500000.0,
        value=3000.0,
        step=500.0
    )


# ============================================================
# EXISTING FINANCIAL INFORMATION
# ============================================================

st.header(
    "💳 Existing Financial Information"
)

col1, col2, col3 = st.columns(3)


with col1:

    existing_loans = st.selectbox(
        "Existing Loans",
        [
            "No",
            "Yes"
        ]
    )


with col2:

    current_emi_amount = st.number_input(
        "Current EMI Amount (₹)",
        min_value=0.0,
        max_value=1000000.0,
        value=0.0,
        step=500.0
    )


with col3:

    credit_score = st.number_input(
        "Credit Score",
        min_value=300,
        max_value=900,
        value=750,
        step=1
    )


col1, col2 = st.columns(2)


with col1:

    bank_balance = st.number_input(
        "Bank Balance (₹)",
        min_value=0.0,
        max_value=100000000.0,
        value=100000.0,
        step=5000.0
    )


with col2:

    emergency_fund = st.number_input(
        "Emergency Fund (₹)",
        min_value=0.0,
        max_value=100000000.0,
        value=100000.0,
        step=5000.0
    )


# ============================================================
# LOAN REQUEST
# ============================================================

st.header(
    "🏦 Loan / EMI Request"
)

col1, col2, col3 = st.columns(3)


with col1:

    emi_scenario = st.selectbox(
        "EMI Scenario",
        [
            "E-commerce Shopping EMI",
            "Education EMI",
            "Home Appliances EMI",
            "Personal Loan EMI",
            "Vehicle EMI"
        ]
    )


with col2:

    requested_amount = st.number_input(
        "Requested Amount (₹)",
        min_value=1000.0,
        max_value=10000000.0,
        value=100000.0,
        step=5000.0
    )


with col3:

    requested_tenure = st.number_input(
        "Requested Tenure (Months)",
        min_value=3,
        max_value=120,
        value=24,
        step=1
    )


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def calculate_features():

    total_monthly_expenses = (
        monthly_rent
        + school_fees
        + college_fees
        + travel_expenses
        + groceries_utilities
        + other_monthly_expenses
    )

    total_monthly_obligations = (
        total_monthly_expenses
        + current_emi_amount
    )

    safe_salary = max(
        monthly_salary,
        1.0
    )

    debt_to_income_ratio = (
        current_emi_amount
        / safe_salary
    )

    expense_to_income_ratio = (
        total_monthly_expenses
        / safe_salary
    )

    obligation_to_income_ratio = (
        total_monthly_obligations
        / safe_salary
    )

    available_monthly_income = (
        monthly_salary
        - total_monthly_obligations
    )

    affordability_ratio = (
        available_monthly_income
        / safe_salary
    )

    # Approximate EMI calculation.
    # This is used only to reproduce the engineered input
    # structure required by the trained model.

    annual_interest_rate = 0.12

    monthly_rate = (
        annual_interest_rate / 12
    )

    if monthly_rate > 0:

        estimated_requested_emi = (
            requested_amount
            * monthly_rate
            * (1 + monthly_rate) ** requested_tenure
            /
            (
                (1 + monthly_rate) ** requested_tenure
                - 1
            )
        )

    else:

        estimated_requested_emi = (
            requested_amount
            / requested_tenure
        )

    loan_to_income_ratio = (
        requested_amount
        / max(monthly_salary * 12, 1.0)
    )

    projected_emi_burden = (
        current_emi_amount
        + estimated_requested_emi
    )

    projected_emi_to_income_ratio = (
        projected_emi_burden
        / safe_salary
    )

    emergency_fund_coverage_months = (
        emergency_fund
        / max(
            total_monthly_expenses,
            1.0
        )
    )

    bank_balance_to_salary_ratio = (
        bank_balance
        / safe_salary
    )

    employment_stability_score = min(
        years_of_employment / 10.0,
        1.0
    )

    credit_score_normalized = (
        (credit_score - 300)
        / 600
    )

    dependent_ratio = (
        dependents
        / max(family_size, 1)
    )

    # --------------------------------------------------------
    # Credit risk
    # --------------------------------------------------------

    if credit_score >= 750:

        credit_risk_level = "Very_Low_Risk"

    elif credit_score >= 700:

        credit_risk_level = "Low_Risk"

    elif credit_score >= 650:

        credit_risk_level = "Moderate_Risk"

    elif credit_score >= 600:

        credit_risk_level = "High_Risk"

    else:

        credit_risk_level = "Very_High_Risk"

    # --------------------------------------------------------
    # Financial stress
    # --------------------------------------------------------

    stress_ratio = (
        total_monthly_obligations
        / safe_salary
    )

    if stress_ratio < 0.30:

        financial_stress_indicator = "Low_Stress"

    elif stress_ratio < 0.50:

        financial_stress_indicator = "Moderate_Stress"

    elif stress_ratio < 0.70:

        financial_stress_indicator = "High_Stress"

    else:

        financial_stress_indicator = "Very_High_Stress"

    return {

        "age": age,
        "gender": gender,
        "marital_status": marital_status,
        "education": education,
        "monthly_salary": monthly_salary,
        "employment_type": employment_type,
        "years_of_employment": years_of_employment,
        "company_type": company_type,
        "house_type": house_type,
        "monthly_rent": monthly_rent,
        "family_size": family_size,
        "dependents": dependents,
        "school_fees": school_fees,
        "college_fees": college_fees,
        "travel_expenses": travel_expenses,
        "groceries_utilities": groceries_utilities,
        "other_monthly_expenses": other_monthly_expenses,
        "existing_loans": existing_loans,
        "current_emi_amount": current_emi_amount,
        "credit_score": credit_score,
        "bank_balance": bank_balance,
        "emergency_fund": emergency_fund,
        "emi_scenario": emi_scenario,
        "requested_amount": requested_amount,
        "requested_tenure": requested_tenure,

        "total_monthly_expenses":
            total_monthly_expenses,

        "total_monthly_obligations":
            total_monthly_obligations,

        "debt_to_income_ratio":
            debt_to_income_ratio,

        "expense_to_income_ratio":
            expense_to_income_ratio,

        "obligation_to_income_ratio":
            obligation_to_income_ratio,

        "available_monthly_income":
            available_monthly_income,

        "affordability_ratio":
            affordability_ratio,

        "estimated_requested_emi":
            estimated_requested_emi,

        "loan_to_income_ratio":
            loan_to_income_ratio,

        "projected_emi_burden":
            projected_emi_burden,

        "projected_emi_to_income_ratio":
            projected_emi_to_income_ratio,

        "emergency_fund_coverage_months":
            emergency_fund_coverage_months,

        "bank_balance_to_salary_ratio":
            bank_balance_to_salary_ratio,

        "employment_stability_score":
            employment_stability_score,

        "credit_score_normalized":
            credit_score_normalized,

        "credit_risk_level":
            credit_risk_level,

        "dependent_ratio":
            dependent_ratio,

        "financial_stress_indicator":
            financial_stress_indicator
    }


# ============================================================
# PREDICTION BUTTON
# ============================================================

st.divider()

predict_button = st.button(
    "🚀 Predict EMI & Analyse Risk",
    type="primary",
    use_container_width=True
)


if predict_button:

    try:

        # ----------------------------------------------------
        # Create input
        # ----------------------------------------------------

        input_data = calculate_features()

        input_df = pd.DataFrame(
            [input_data]
        )

        # ----------------------------------------------------
        # Ensure exact preprocessor columns
        # ----------------------------------------------------

        required_columns = list(
            preprocessor.feature_names_in_
        )

        input_df = input_df[
            required_columns
        ]

        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        transformed_input = (
            preprocessor.transform(
                input_df
            )
        )

        # ----------------------------------------------------
        # Verify 74 features
        # ----------------------------------------------------

        if transformed_input.shape[1] != 74:

            st.error(
                "Unexpected transformed feature count."
            )

            st.write(
                "Expected: 74"
            )

            st.write(
                "Received:",
                transformed_input.shape[1]
            )

            st.stop()

        # ----------------------------------------------------
        # Regression
        # ----------------------------------------------------

        predicted_emi = (
            regression_model.predict(
                transformed_input
            )[0]
        )

        predicted_emi = max(
            float(predicted_emi),
            0.0
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        rf_prediction = (
            rf_model.predict(
                transformed_input
            )[0]
        )

        xgb_prediction = (
            xgb_model.predict(
                transformed_input
            )[0]
        )

        # ----------------------------------------------------
        # Decode classification
        # ----------------------------------------------------

        try:

            rf_label = label_encoder.inverse_transform(
                [rf_prediction]
            )[0]

        except Exception:

            rf_label = str(
                rf_prediction
            )


        try:

            xgb_label = label_encoder.inverse_transform(
                [xgb_prediction]
            )[0]

        except Exception:

            xgb_label = str(
                xgb_prediction
            )

        # ----------------------------------------------------
        # Display prediction
        # ----------------------------------------------------

        st.success(
            "Prediction completed successfully!"
        )

        st.header(
            "📊 Prediction Results"
        )

        col1, col2, col3 = st.columns(3)

        with col1:

            st.metric(
                "Predicted Maximum Monthly EMI",
                f"₹{predicted_emi:,.2f}"
            )

        with col2:

            st.metric(
                "Estimated Requested EMI",
                f"₹{input_data['estimated_requested_emi']:,.2f}"
            )

        with col3:

            st.metric(
                "Available Monthly Income",
                f"₹{input_data['available_monthly_income']:,.2f}"
            )

        # ----------------------------------------------------
        # Classification results
        # ----------------------------------------------------

        st.header(
            "🏦 Loan Risk Analysis"
        )

        col1, col2 = st.columns(2)

        with col1:

            st.subheader(
                "Random Forest"
            )

            st.info(
                str(rf_label)
            )

        with col2:

            st.subheader(
                "XGBoost"
            )

            st.info(
                str(xgb_label)
            )

        # ----------------------------------------------------
        # Financial indicators
        # ----------------------------------------------------

        st.header(
            "📈 Financial Indicators"
        )

        col1, col2, col3, col4 = st.columns(4)

        with col1:

            st.metric(
                "Debt-to-Income",
                f"{input_data['debt_to_income_ratio']:.2%}"
            )

        with col2:

            st.metric(
                "Expense-to-Income",
                f"{input_data['expense_to_income_ratio']:.2%}"
            )

        with col3:

            st.metric(
                "Credit Score",
                f"{credit_score}"
            )

        with col4:

            st.metric(
                "Emergency Coverage",
                f"{input_data['emergency_fund_coverage_months']:.1f} months"
            )

        # ----------------------------------------------------
        # User-friendly interpretation
        # ----------------------------------------------------

        st.header(
            "💡 Financial Interpretation"
        )

        if (
            predicted_emi
            <= input_data["available_monthly_income"]
        ):

            st.success(
                "The predicted maximum EMI is within "
                "the applicant's available monthly income."
            )

        else:

            st.warning(
                "The predicted maximum EMI is higher than "
                "the applicant's currently available monthly income."
            )

        if (
            input_data["debt_to_income_ratio"]
            <= 0.30
        ):

            st.success(
                "Existing EMI burden is relatively low."
            )

        elif (
            input_data["debt_to_income_ratio"]
            <= 0.50
        ):

            st.warning(
                "Existing EMI burden is moderate."
            )

        else:

            st.error(
                "Existing EMI burden is high."
            )

        # ----------------------------------------------------
        # Technical information
        # ----------------------------------------------------

        with st.expander(
            "🔍 Technical Model Information"
        ):

            st.write(
                "Input features:",
                len(required_columns)
            )

            st.write(
                "Transformed features:",
                transformed_input.shape[1]
            )

            st.write(
                "Regression model:",
                type(regression_model).__name__
            )

            st.write(
                "Random Forest:",
                type(rf_model).__name__
            )

            st.write(
                "XGBoost:",
                type(xgb_model).__name__
            )

    except Exception as e:

        st.error(
            "Prediction failed."
        )

        st.exception(e)
'''

app_path.write_text(
    app_code,
    encoding="utf-8"
)

print("=" * 70)
print("STREAMLIT APPLICATION CREATED")
print("=" * 70)

print()
print("File:")
print(app_path)

print()
print("Exists:", app_path.exists())

print()
print("Size:", app_path.stat().st_size, "bytes")

STREAMLIT APPLICATION CREATED

File:
C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app\app.py

Exists: True

Size: 22753 bytes


In [18]:
# ============================================================
# STEP 11
# CHECK APP.PY SYNTAX
# ============================================================

import py_compile

app_file = (
    r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI"
    r"\streamlit_app\app.py"
)

try:

    py_compile.compile(
        app_file,
        doraise=True
    )

    print(
        "✅ app.py syntax is correct."
    )

except Exception as e:

    print(
        "❌ Syntax error found:"
    )

    print(e)

✅ app.py syntax is correct.


In [19]:
# ==========================================
# STEP 1
# VERIFY STREAMLIT APPLICATION FILES
# ==========================================

import os

APP_DIR = r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app"

print("=" * 70)
print("STREAMLIT APPLICATION FILE CHECK")
print("=" * 70)

required_files = [
    "app.py",
    "best_emi_regression_model.pkl",
    "classification_preprocessor.pkl",
    "random_forest_classifier.pkl",
    "xgboost_classifier.pkl",
    "xgb_label_encoder.pkl"
]

all_files_ok = True

for file_name in required_files:

    file_path = os.path.join(
        APP_DIR,
        file_name
    )

    if os.path.exists(file_path):

        print("✅", file_name)

    else:

        print("❌ MISSING:", file_name)
        all_files_ok = False

print("=" * 70)

if all_files_ok:

    print("✅ ALL REQUIRED STREAMLIT FILES ARE AVAILABLE")

else:

    print("❌ SOME REQUIRED FILES ARE MISSING")

STREAMLIT APPLICATION FILE CHECK
✅ app.py
✅ best_emi_regression_model.pkl
✅ classification_preprocessor.pkl
✅ random_forest_classifier.pkl
✅ xgboost_classifier.pkl
✅ xgb_label_encoder.pkl
✅ ALL REQUIRED STREAMLIT FILES ARE AVAILABLE


In [20]:
# ==========================================
# STEP 2
# START STREAMLIT APPLICATION
# ==========================================

import subprocess
import os

APP_DIR = r"C:\Users\MAHESH KUMAR\OneDrive\Desktop\EMI\streamlit_app"

app_path = os.path.join(
    APP_DIR,
    "app.py"
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        app_path,
        "--server.headless=true"
    ],
    cwd=APP_DIR
)

print("=" * 70)
print("STREAMLIT APPLICATION STARTED")
print("=" * 70)

print("Application URL:")
print("http://localhost:8501")

print("=" * 70)

STREAMLIT APPLICATION STARTED
Application URL:
http://localhost:8501


In [1]:
!jupyter nbconvert --to script EMI_Streamlit_App.ipynb

This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr

[NbConvertApp] WARNING | pattern 'EMI_Streamlit_App.ipynb' matched no files
